# Phase analysis: RQ2

*How does this impact differ between phases of application execution?*

Feeds: **§6.3 Phases (RQ2)** (startup versus workload, durations, Figure 2).

Independent of the other notebooks. `teardown_joules` is not a measured phase and is not analysed: the profiler stops
asynchronously and USB power is restored while it is still sampling, so that window contains charge current.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import thesis_lib as T

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
REF = T.reference()
for d in (T.TABLES, T.FIGURES, T.SUPERVISOR):
    d.mkdir(exist_ok=True)
T.figure_style()
NB = "notebooks/Phase_Analysis.ipynb"

In [2]:
raw = T.load_raw()
print(f"loaded {len(raw)} rows from {len(T.RUN_TABLES)} run tables")
df, chain = T.apply_filters(raw)
series = df[df["batch"].isin(T.SERIES)].copy()
print(f"series cells entering the analysis: {len(series)}")

loaded 4800 rows from 4 run tables
series cells entering the analysis: 2647


## §6.3 (RQ2): Startup versus workload, per technique

The same estimator as the headline table (Headline_Effects), recomputed here so this notebook stands alone.

In [3]:
rows = []
KEY = {v: k for k, v in T.DISPLAY.items()}
for arm in T.ARMS:
    r = {"technique": T.name(arm)}
    for w, col in T.WINDOWS.items():
        r[w] = T.delta_e(T.pairs(T.phase_set(series, w), arm, col))
        T.check(f"{T.name(arm):<9}{w:<9} delta-E %", r[w], REF["headline_pooled_series"][f"{T.short(arm)}_{w}"]["delta_pct"])
    rows.append(r)
pc = pd.DataFrame(rows)[["technique", "startup", "workload"]]
pc.to_csv(T.TABLES / "phase_contrast.csv", index=False)
T.register("tables/phase_contrast.csv", "REPO", "6.3 RQ2 (phases)", "passive vs active delta-E per technique", NB)
print(pc.round(2).to_string(index=False))

  Bangcle  workload  delta-E %: -0.0187   (expected -0.0187)  OK
  Bangcle  startup   delta-E %: +0.0977   (expected +0.0977)  OK
  Allatori workload  delta-E %: -0.4364   (expected -0.4364)  OK
  Allatori startup   delta-E %: -0.2944   (expected -0.2944)  OK
  freeRASP workload  delta-E %: +4.0616   (expected +4.0616)  OK
  freeRASP startup   delta-E %: +31.7905   (expected +31.7905)  OK
technique  startup  workload
  Bangcle     0.10     -0.02
 Allatori    -0.29     -0.44
 freeRASP    31.79      4.06


## §6.3 (RQ2): Phase durations and mean power

Same pairing as the energy analysis, applied to the window durations and to mean power in the window (joules ÷ seconds per cell).
If a technique made its startup more expensive only by making it longer, the duration would rise with the energy.

In [4]:
series["startup_power_w"] = series["pre_workload_joules"] / series["duration_pre_workload_s"]
series["workload_power_w"] = series["workload_joules"] / series["duration_workload_s"]
rows = []
for arm in T.ARMS:
    # key: the reference implementation's name for the window (startup/workload); label: the phase name a reader sees
    for key_, label, col in (("startup", "passive-phase length", "duration_pre_workload_s"), ("workload", "active-phase length", "duration_workload_s"),
                             ("startup", "passive-phase mean power", "startup_power_w"), ("workload", "active-phase mean power", "workload_power_w")):
        p = T.pairs(T.phase_set(series, key_), arm, col)            # the passive phase leaves out medtimer
        m, lo, hi, _ = T.ci_naive(p["delta_pct"]); ms, los, his, _ = T.ci_naive(p["delta_abs"])
        rows.append(dict(technique=T.name(arm), measure=label, n_pairs=len(p), delta_pct=m, ci_lo=lo, ci_hi=hi,
                         delta_abs=ms, abs_ci_lo=los, abs_ci_hi=his, pairs_positive=int((p["delta_pct"] > 0).sum())))
        if "length" in label:
            key = f"{T.short(arm)}_{key_}"
            T.check(f"{T.name(arm):<9}{label:<18} %", m, REF["phase_duration_deltas"][key]["delta_pct"])
            T.check(f"{T.name(arm):<9}{label:<18} s", ms, REF["phase_duration_deltas"][key]["delta_s"])
            T.record(f"duration.{T.short(arm)}.{key_}", delta_s=ms, delta_pct=m, ci95_pct=[lo, hi],
                     pairs_positive=f"{int((p['delta_pct'] > 0).sum())} of {len(p)}")
dur = pd.DataFrame(rows)
dur.to_csv(T.TABLES / "phase_durations_and_power.csv", index=False)
T.register("tables/phase_durations_and_power.csv", "REPO", "6.3 RQ2 (phases)", "paired change in window duration and mean power, per technique", NB)
print(dur.round(3).to_string(index=False))

  Bangcle  passive-phase length %: -0.0229   (expected -0.0229)  OK
  Bangcle  passive-phase length s: -0.0101   (expected -0.0101)  OK
  Bangcle  active-phase length %: +0.1327   (expected +0.1327)  OK
  Bangcle  active-phase length s: +0.1447   (expected +0.1447)  OK
  Allatori passive-phase length %: +0.2353   (expected +0.2353)  OK
  Allatori passive-phase length s: +0.0824   (expected +0.0824)  OK
  Allatori active-phase length %: +0.1467   (expected +0.1467)  OK
  Allatori active-phase length s: +0.1513   (expected +0.1513)  OK
  freeRASP passive-phase length %: -3.6490   (expected -3.6490)  OK
  freeRASP passive-phase length s: -1.2721   (expected -1.2721)  OK
  freeRASP active-phase length %: +0.0041   (expected +0.0041)  OK
  freeRASP active-phase length s: -0.0137   (expected -0.0137)  OK


technique                  measure  n_pairs  delta_pct  ci_lo  ci_hi  delta_abs  abs_ci_lo  abs_ci_hi  pairs_positive
  Bangcle     passive-phase length       69     -0.023 -1.097  1.051     -0.010     -0.421      0.401              34
  Bangcle      active-phase length       69      0.133 -0.200  0.465      0.145     -0.242      0.531              39
  Bangcle passive-phase mean power       69      0.356 -0.936  1.649      0.005     -0.032      0.043              38
  Bangcle  active-phase mean power       69     -0.147 -0.950  0.656     -0.008     -0.033      0.018              36
 Allatori     passive-phase length       36      0.235 -0.841  1.312      0.082     -0.294      0.458              22
 Allatori      active-phase length       45      0.147 -0.044  0.338      0.151     -0.066      0.368              25
 Allatori passive-phase mean power       36     -0.533 -2.180  1.114     -0.017     -0.063      0.030              15
 Allatori  active-phase mean power       45     -0.602 -

## §6.3 (RQ2): What the startup window contains, and the launch measured directly

**The startup window does not contain the cold launch.** Android Runner launches the application (`monkey`), runs its
`after_launch` hook, and only then starts the profilers. The startup window runs from the first battery sample to the first
scenario. So the application's process start and first display come *before* energy is measured. Inside the window are Appium's
session (its UiAutomator2 server, then a warm start of the already-running application) and the application's first-use
screens.

**The launch itself is in the system log.** For every captured cell, `launch_timing_series.csv` (from the raw per-cell
logcat; `tools/extract_launch_timing.py`, provenance beside it) gives the time to initial display of the cold launch
(`ActivityTaskManager: Displayed ...`) and when each event happened relative to the first battery sample. Matched pairs as
everywhere else: group means per (application, device, batch), then the mean over pairs of the percentage difference. Not one
of the six headline contrasts: reported with its interval and minimum detectable effect, without a p-value.

Last, the startup window's resolution in absolute terms: its minimum detectable effect in joules and in seconds, beside its mean.

In [5]:
key = ["batch", "phone", "app", "variant", "rep"]
lt = pd.read_csv(T.DATA / "inputs" / "launch_timing_series.csv", dtype={"rep": str})
lt = lt.merge(series[key + ["duration_pre_workload_s"]], on=key, how="inner")        # analysed cells only
# Appium's session start: the second activity start when it is a MAIN intent before the first scenario (six applications)
lt["appium_session_rel_s"] = lt["appium_start_rel_s"].where((lt["appium_start_action"] == "MAIN")
                                                            & (lt["appium_start_rel_s"] < lt["duration_pre_workload_s"]))
gap = lt["first_display_rel_s"] - lt["proc_start_rel_s"]
assert (lt["first_display_rel_s"].dropna() < 0).all() and (gap.dropna() > 0).all(), "a first display is not the cold launch"
inside = int((lt["proc_start_rel_s"] > 0).sum())
print(f"{len(lt)} analysed cells with a log capture; {int(lt['ttid_ms'].notna().sum())} with a cold-launch display time; "
      f"application process started inside the passive phase in {inside} of {int(lt['proc_start_rel_s'].notna().sum())}")

# 1. what the startup window contains (seconds from the start of energy measurement)
events = [("cold launch: application process starts", "proc_start_rel_s"),
          ("cold launch: first display", "first_display_rel_s"),
          ("Appium's UiAutomator2 server starts", "uia2_start_rel_s"),
          ("Appium sends the launch intent again (session start, five of the passive phase's eight applications; no new process)",
           "appium_session_rel_s"),
          ("first scenario: the passive phase ends", "duration_pre_workload_s")]
rows = []
for label, col in events:
    src = lt if col in ("proc_start_rel_s", "first_display_rel_s") else T.phase_set(lt, "passive")   # the launch keeps medtimer
    for v in ("baseline_debug", "bangcle_debug", "allatori_debug", "freerasp_debug"):
        s = src.loc[src["variant"] == v, col].dropna()
        rows.append({"event": label, "technique": T.name(v), "cells": len(s), "median_s": s.median(),
                     "p5_s": s.quantile(0.05), "p95_s": s.quantile(0.95)})
comp = pd.DataFrame(rows)
comp.to_csv(T.TABLES / "startup_window_composition.csv", index=False)
T.register("tables/startup_window_composition.csv", "REPO", "5 Execution / 6.3 RQ2 (what the passive phase contains)",
           "when the cold launch, Appium's session and the first scenario happen, relative to the start of energy measurement", NB)
print(comp[comp["technique"].isin(["Control", "Bangcle"])].round(1).to_string(index=False))
ctl = comp[comp["technique"] == "Control"].set_index("event")["median_s"]
T.record("startup_window.composition_control_median_s", process_start=ctl.iloc[0], first_display=ctl.iloc[1],
         uia2_server_start=ctl.iloc[2], appium_session_start=ctl.iloc[3], window_closes=ctl.iloc[4],
         process_starts_inside_window=f"{inside} of {int(lt['proc_start_rel_s'].notna().sum())}")

# 2. the launch: time to initial display, technique against control, over all devices and per device
g = lt.dropna(subset=["ttid_ms"]).groupby(["app", "phone", "batch", "variant"])["ttid_ms"].mean().unstack("variant")
rows = []
for arm in T.ARMS:
    d_all = g[[arm, "baseline_debug"]].dropna()
    for dev in ("all devices", "pixel3", "pixel6", "pixel9"):
        d = d_all if dev == "all devices" else d_all[d_all.index.get_level_values("phone") == dev]
        pct = 100 * (d[arm] - d["baseline_debug"]) / d["baseline_debug"]
        ms = d[arm] - d["baseline_debug"]
        m, lo, hi, _ = T.ci_naive(pct)
        ma, loa, hia, _ = T.ci_naive(ms)
        rows.append({"technique": T.name(arm), "device": dev if dev == "all devices" else T.PHONE_DISPLAY[dev],
                     "pairs": len(d), "control_mean_ms": d["baseline_debug"].mean(), "treated_mean_ms": d[arm].mean(),
                     "delta_pct": m, "ci_lo": lo, "ci_hi": hi, "delta_ms": ma, "ms_ci_lo": loa, "ms_ci_hi": hia,
                     "mde_pct": T.mde(pct), "mde_ms": T.mde(ms), "pairs_longer": int((pct > 0).sum())})
lt_tab = pd.DataFrame(rows)
for arm in T.ARMS:
    t = lt_tab[lt_tab["technique"] == T.name(arm)].set_index("device")
    a = t.loc["all devices"]
    assert t.drop(index="all devices")["pairs"].sum() == a["pairs"], "per-device pairs do not add up"
    T.record(f"launch_time.{T.short(arm)}", delta_pct=a["delta_pct"], ci95=[a["ci_lo"], a["ci_hi"]], delta_ms=a["delta_ms"],
             ci95_ms=[a["ms_ci_lo"], a["ms_ci_hi"]], mde_pct=a["mde_pct"], mde_ms=a["mde_ms"], pairs=int(a["pairs"]),
             control_mean_ms=a["control_mean_ms"], pairs_longer=int(a["pairs_longer"]),
             per_device_delta_pct_pixel3_6_9=[t.loc[T.PHONE_DISPLAY[p], "delta_pct"] for p in ("pixel3", "pixel6", "pixel9")],
             method="paired t over matched pairs; exploratory, no p")
lt_tab.to_csv(T.TABLES / "launch_time.csv", index=False)
T.register("tables/launch_time.csv", "REPO", "6.3 RQ2 (the launch)",
           "cold-launch time to initial display (system log), each technique against its control, over all devices and per device: "
           "pairs, %, ms, 95% CI, MDE, pairs longer", NB)
print(); print(lt_tab.round(2).to_string(index=False))

# 3. the startup window's resolution in absolute terms
rows = []
for arm in T.ARMS:
    sp = T.phase_set(series, "passive")            # the passive phase leaves out medtimer
    pe = T.pairs(sp, arm, "pre_workload_joules")
    pdur = T.pairs(sp, arm, "duration_pre_workload_s")
    rows.append({"technique": T.name(arm), "pairs": len(pe), "window_mean_j": pe["baseline"].mean(),
                 "mde_pct": T.mde(pe["delta_pct"]), "mde_j": T.mde(pe["delta_abs"]),
                 "window_mean_s": pdur["baseline"].mean(), "duration_mde_pct": T.mde(pdur["delta_pct"]),
                 "duration_mde_s": T.mde(pdur["delta_abs"])})
res = pd.DataFrame(rows)
mde_ref = pd.read_csv(T.TABLES / "minimum_detectable_effect.csv")
for r in res.itertuples():
    ref = mde_ref[(mde_ref["Technique"] == r.technique) & (mde_ref["Window"] == T.WINDOW_DISPLAY["startup"])]["MDE (±%)"].iloc[0]
    assert abs(round(r.mde_pct, 2) - ref) < 0.006, "differs from minimum_detectable_effect.csv"
res.to_csv(T.TABLES / "startup_window_resolution.csv", index=False)
T.register("tables/startup_window_resolution.csv", "REPO", "6.3 RQ2 / Threats (resolution)",
           "the passive phase's mean and its minimum detectable effect in percent, joules and seconds, per technique", NB)
print(); print(res.round(2).to_string(index=False))
for r in res.itertuples():
    T.record(f"startup_window.resolution.{r.technique.lower()}", window_mean_j=r.window_mean_j, mde_j=r.mde_j,
             window_mean_s=r.window_mean_s, mde_s=r.duration_mde_s, mde_pct=r.mde_pct)

2645 analysed cells with a log capture; 2645 with a cold-launch display time; application process started inside the passive phase in 0 of 2645
                                                                                                               event technique  cells  median_s  p5_s  p95_s
                                                                             cold launch: application process starts   Control    760      -7.5  -8.4   -6.9
                                                                             cold launch: application process starts   Bangcle    676      -7.3  -8.5   -6.9
                                                                                          cold launch: first display   Control    760      -6.9  -7.9   -6.1
                                                                                          cold launch: first display   Bangcle    676      -6.5  -7.7   -4.7
                                                                       


technique      device  pairs  control_mean_ms  treated_mean_ms  delta_pct  ci_lo  ci_hi  delta_ms  ms_ci_lo  ms_ci_hi  mde_pct  mde_ms  pairs_longer
  Bangcle all devices     69           600.32          1024.73      93.47  77.61 109.33    424.42    341.29    507.54    22.59  118.41            63
  Bangcle     Pixel 3     21           899.65          1644.06     115.43  89.06 141.80    744.40    581.44    907.37    37.24  230.14            21
  Bangcle     Pixel 6     24           445.22           729.04      89.13  57.86 120.39    283.82    180.00    387.64    44.23  146.86            21
  Bangcle     Pixel 9     24           493.49           778.52      78.60  52.60 104.60    285.02    192.80    377.25    36.78  130.46            21
 Allatori all devices     45           719.32           717.11       0.09  -1.28   1.46     -2.21    -16.66     12.25     1.95   20.55            19
 Allatori     Pixel 3     15          1020.38          1003.95       0.06  -2.40   2.53    -16.42    -45.

## Not used in the thesis: the launch's energy, estimated from in-window relaunches (exploratory)

**Not used in the thesis** (decision of 2026-10-03): one application, nine pairs, relaunches rather than first launches, and a fuel gauge whose cited validation covers only windows of tens of seconds. It is kept here, with its checks, for the replication package. The fuel gauge's resolution (`fuel_gauge_resolution.csv`, below) is used, in Threats.

The first launch of every cell precedes the energy measurement, so its energy was never sampled. But some launches happened
*while* energy was being measured: poetskingdom's workload terminates and relaunches the application by design, in every cell
and for all four builds (`appium_android_tests/poetskingdom/scenarios.py`, `_create_poem`). `inwindow_launches_series.csv`
(`tools/extract_inwindow_launches.py`; provenance beside it) gives, for each such cold launch, its time to initial display and
the energy the battery samples show over it.

Per launch:
* **energy during the launch**: from the launch request to the first frame. The window spans only a few battery samples, so
  the power is interpolated at both of its edges;
* **the same plus 1 s**: the fuel gauge's reading changes only a few times a second (`fuel_gauge_resolution.csv`), so part of
  a launch's draw shows a little later;
* **above the pre-launch level**: minus what the phone drew in the 2 s before, over the same duration. That is the launch's own
  work, without the screen and the rest of the phone, which draw power during the extra time too.

Matched pairs as everywhere else, here (device, batch) for the one application: a group mean per pair, then the paired mean
difference with its interval, and how many of the pairs go up. Exploratory, no p.

**Checks** (the rows marked "check" in the same table, and a second table):
* **the state before the launch**: the power in the 2 s before must not differ between builds, or the comparison is of states;
* **the window shifted 0.33 s earlier and later** (about one fuel-gauge reading on the Pixel 3 and 6): the difference must
  survive wherever the coarse readings fall;
* **Allatori as a negative control**: the same application, obfuscated and launching no slower, must show no difference;
* **resolution** (`fuel_gauge_resolution.csv`): how often the gauge's reading changes, and how many samples and readings one
  launch spans.

**Not done: an extrapolation to every application's first launch.** It would assume that poetskingdom's energy per second of
extra launch time holds for the other eight applications, and that a relaunch costs what the first launch after installation
costs. Neither was measured, so no such number is emitted.

In [6]:
il = pd.read_csv(T.DATA / "inputs" / "inwindow_launches_series.csv", dtype={"rep": str}).merge(series[key], on=key)
print("in-window cold launches in analysed cells, by application and technique:")
print(il.groupby(["app", "variant"]).size().unstack().fillna(0).astype(int).to_string())
pk = il[il["app"] == "poetskingdom"].copy()
assert (pk["launch_number"] >= 2).all(), "a cell's first launch fell inside the measured window"
pk["launch_s"] = pk["ttid_ms"] / 1000
pk["e_above_idle_j"] = pk["e_launch_j"] - pk["p_before_w"] * pk["launch_s"]
pk["e_plus1_above_idle_j"] = pk["e_launch_plus1s_j"] - pk["p_before_w"] * (pk["launch_s"] + 1)
pk["p_launch_w"] = pk["e_launch_j"] / pk["launch_s"]
measures = {"ttid_ms": "launch time (ms)",
            "e_launch_j": "energy during the launch (J)",
            "e_launch_plus1s_j": "energy, launch + 1 s (J)",
            "e_above_idle_j": "energy above the pre-launch level (J)",
            "e_plus1_above_idle_j": "energy above the pre-launch level, launch + 1 s (J)",
            "p_launch_w": "power during the launch (W)",
            "p_before_w": "check: power in the 2 s before the launch (W)",
            "e_launch_early_j": "check: energy during the launch, window 0.33 s earlier (J)",
            "e_launch_late_j": "check: energy during the launch, window 0.33 s later (J)"}
g = pk.groupby(["phone", "batch", "variant"])[list(measures)].mean().unstack("variant")
n_ctl = int((pk["variant"] == "baseline_debug").sum())
rows = []
for arm in T.ARMS:
    for col, label in measures.items():
        d = (g[col][arm] - g[col]["baseline_debug"]).dropna()
        m, lo, hi, _ = T.ci_naive(d)
        rows.append({"technique": T.name(arm), "measure": label, "pairs": len(d), "pairs_higher": int((d > 0).sum()),
                     "control_mean": g[col]["baseline_debug"].mean(), "treated_mean": g[col][arm].mean(),
                     "difference": m, "ci_lo": lo, "ci_hi": hi,
                     "control_launches": n_ctl, "treated_launches": int((pk["variant"] == arm).sum())})
le = pd.DataFrame(rows)
assert (le["pairs"] == 9).all(), "a (device, batch) pair lacks launches of one of its builds"
le.to_csv(T.TABLES / "launch_energy_estimate.csv", index=False)
T.register("tables/launch_energy_estimate.csv", "REPO", "not used in the thesis (replication package only)",
           "launch energy measured on poetskingdom's in-window relaunches, each technique against its control, with the window checks", NB)
with pd.option_context("display.width", 220, "display.max_colwidth", 70):
    print(le.round(2).to_string(index=False))
E, A = "energy during the launch (J)", "energy above the pre-launch level (J)"
for arm in T.ARMS:
    t = le[le["technique"] == T.name(arm)].set_index("measure")
    T.record(f"launch_energy.{T.short(arm)}",
             relaunch_extra_time_ms=t.loc["launch time (ms)", "difference"],
             control_energy_per_launch_j=t.loc[E, "control_mean"], treated_energy_per_launch_j=t.loc[E, "treated_mean"],
             relaunch_extra_energy_j=t.loc[E, "difference"], relaunch_extra_energy_ci95=[t.loc[E, "ci_lo"], t.loc[E, "ci_hi"]],
             relaunch_extra_energy_pairs_higher=f"{t.loc[E, 'pairs_higher']} of {t.loc[E, 'pairs']}",
             relaunch_extra_energy_above_idle_j=t.loc[A, "difference"],
             relaunch_extra_energy_above_idle_ci95=[t.loc[A, "ci_lo"], t.loc[A, "ci_hi"]],
             window_shifted_earlier_later_j=[t.loc["check: energy during the launch, window 0.33 s earlier (J)", "difference"],
                                             t.loc["check: energy during the launch, window 0.33 s later (J)", "difference"]],
             pre_launch_power_diff_w=t.loc["check: power in the 2 s before the launch (W)", "difference"],
             pre_launch_power_diff_ci95=[t.loc["check: power in the 2 s before the launch (W)", "ci_lo"],
                                         t.loc["check: power in the 2 s before the launch (W)", "ci_hi"]],
             basis="poetskingdom's in-window relaunches, 9 (device, batch) pairs; exploratory, no p; not extrapolated; not used in the thesis")

in-window cold launches in analysed cells, by application and technique:
variant       allatori_debug  bangcle_debug  baseline_debug  freerasp_debug
app                                                                        
itsok                      0              1               0               0
poetskingdom             267            267             261             270


technique                                                    measure  pairs  pairs_higher  control_mean  treated_mean  difference  ci_lo  ci_hi  control_launches  treated_launches
  Bangcle                                           launch time (ms)      9             9        340.14        838.73      498.59 348.88 648.29               261               267
  Bangcle                               energy during the launch (J)      9             9          1.28          3.80        2.53   2.32   2.73               261               267
  Bangcle                                   energy, launch + 1 s (J)      9             9          5.54          7.75        2.21   2.04   2.38               261               267
  Bangcle                      energy above the pre-launch level (J)      9             9          0.51          1.91        1.40   1.01   1.78               261               267
  Bangcle        energy above the pre-launch level, launch + 1 s (J)      9             9          2

In [7]:
# What one launch spans: battery samples, and new fuel-gauge readings. The gauge's interval is the median gap between
# changes of the current reading over a whole cell.
cells = pk.drop_duplicates(key)
rows = []
for ph in ("pixel3", "pixel6", "pixel9"):
    u = cells.loc[cells["phone"] == ph, "gauge_update_ms"].dropna()
    row = {"phone": ph, "cells": len(u), "gauge_update_ms_median": u.median(),
           "gauge_update_ms_min": u.min(), "gauge_update_ms_max": u.max()}
    for v in ("baseline_debug", "bangcle_debug"):
        x = pk[(pk["phone"] == ph) & (pk["variant"] == v)]
        row[f"launch_ms_median_{T.short(v)}"] = x["ttid_ms"].median()
        row[f"samples_in_launch_median_{T.short(v)}"] = x["samples_in_launch"].median()
        row[f"gauge_updates_in_launch_median_{T.short(v)}"] = x["gauge_updates_in_launch"].median()
    rows.append(row)
fg = pd.DataFrame(rows)
fg.to_csv(T.TABLES / "fuel_gauge_resolution.csv", index=False)
T.register("tables/fuel_gauge_resolution.csv", "REPO", "Threats (the launch: the instrument's resolution)",
           "per device: how often the fuel gauge's current reading changes, and how many battery samples and gauge readings one launch spans", NB)
print(fg.to_string(index=False))
T.record("launch_energy.fuel_gauge", **{f"update_ms_{r.phone}": r.gauge_update_ms_median for r in fg.itertuples()},
         basis="median interval between changes of the current reading over a cell, poetskingdom cells")

 phone  cells  gauge_update_ms_median  gauge_update_ms_min  gauge_update_ms_max  launch_ms_median_baseline  samples_in_launch_median_baseline  gauge_updates_in_launch_median_baseline  launch_ms_median_bangcle  samples_in_launch_median_bangcle  gauge_updates_in_launch_median_bangcle
pixel3    115                   334.0                  327                  337                      442.0                                4.0                                      1.0                    1196.0                              12.0                                     3.0
pixel6    120                   317.0                  316                  326                      275.0                                3.0                                      1.0                     664.0                               7.0                                     2.0
pixel9    120                   217.0                  213                  218                      311.0                                3.0          

## §6.3 (RQ2): freeRASP's startup cost behaves like a fixed cost

Per application: mean control joules and mean freeRASP joules over that application's matched groups, and the difference in
**joules** and in **percent**. A fixed cost predicts an overhead that is stable in joules and varies in percent with the size of
the control; a proportional cost predicts the reverse. The evidence is therefore which of the two varies less across
applications: the coefficient of variation in joules against that in percent.

**Startup:** in eight of nine applications the overhead lies in a narrow joule band, and it varies far less in joules than in
percent. medtimer is the exception (a much larger control, less than half the overhead) and is reported, not removed: the
band and the CVs are given for the eight, with medtimer stated beside them.
**Workload, the counter-check:** the overhead varies about as much in joules as in percent, so the fixed-cost reading does not
extend to the workload window. Exploratory; no test is attached, and no correlation is reported for this claim.

In [8]:
from scipy import stats as _st
rows, summ = [], []
for w, col in T.WINDOWS.items():
    p = T.pairs(series, "freerasp_debug", col)
    per = p.groupby("app").agg(control_j=("baseline", "mean"), freerasp_j=("treated", "mean"))
    per["delta_j"] = per["freerasp_j"] - per["control_j"]
    per["delta_pct"] = 100 * per["delta_j"] / per["control_j"]
    per["window"] = w
    rows.append(per.reset_index())
    for subset, d in (("all nine", per), ("eight, without medtimer", per.drop(index="medtimer"))):
        summ.append(dict(window=w, subset=subset, n_apps=len(d), mean_delta_j=d["delta_j"].mean(),
                         min_delta_j=d["delta_j"].min(), max_delta_j=d["delta_j"].max(),
                         cv_joules_pct=100 * d["delta_j"].std(ddof=1) / abs(d["delta_j"].mean()),
                         cv_percent_pct=100 * d["delta_pct"].std(ddof=1) / abs(d["delta_pct"].mean())))
    # verified against the independent implementation, not reported: over the phase's applications (the passive
    # phase without medtimer, as in the reference implementation)
    per_ph = per if w == "workload" else per.drop(index=list(T.PASSIVE_EXCLUDED_APPS))
    r_ph = _st.pearsonr(per_ph["control_j"], per_ph["delta_j"])[0]
    T.check(f"{w:<9} baseline-vs-overhead correlation, the phase's apps", r_ph,
            REF["freerasp_scaling"][w]["corr_baseline_vs_abs_delta"], tol=0.001, show=False)
ft = pd.concat(rows); ft.to_csv(T.TABLES / "freerasp_fixed_cost_per_app.csv", index=False)
T.register("tables/freerasp_fixed_cost_per_app.csv", "REPO", "6.3 RQ2 (phases)", "freeRASP overhead per application in joules and percent, both windows", NB)
fs = pd.DataFrame(summ); fs.to_csv(T.TABLES / "freerasp_fixed_cost_summary.csv", index=False)
T.register("tables/freerasp_fixed_cost_summary.csv", "REPO", "6.3 RQ2 (phases)", "joule band and CV in joules vs CV in percent, passive-phase and workload, with and without medtimer", NB)
print(ft[ft["window"] == "startup"].round(2).to_string(index=False)); print(); print(fs.round(1).to_string(index=False))
s8 = fs.query("window == 'startup' and subset == 'eight, without medtimer'").iloc[0]
w9 = fs.query("window == 'workload' and subset == 'all nine'").iloc[0]
med = ft.query("window == 'startup' and app == 'medtimer'").iloc[0]
print(f"\nstartup, eight apps: {s8.min_delta_j:.1f}-{s8.max_delta_j:.1f} J; CV {s8.cv_joules_pct:.1f}% in joules vs "
      f"{s8.cv_percent_pct:.1f}% in percent; medtimer {med.delta_j:.1f} J")
print(f"workload counter-check, nine apps: CV {w9.cv_joules_pct:.0f}% in joules vs {w9.cv_percent_pct:.0f}% in percent")
T.record("fixed_cost.freerasp.startup", band_j=[s8.min_delta_j, s8.max_delta_j], n_apps_in_band=int(s8.n_apps),
         cv_joules_pct=s8.cv_joules_pct, cv_percent_pct=s8.cv_percent_pct, exception="medtimer", medtimer_delta_j=med.delta_j,
         note="exploratory; no test; no correlation reported")
T.record("fixed_cost.freerasp.workload_countercheck", n_apps=int(w9.n_apps), cv_joules_pct=w9.cv_joules_pct,
         cv_percent_pct=w9.cv_percent_pct, note="fixed-cost reading does not extend to the active phase")

  workload  baseline-vs-overhead correlation, the phase's apps: matches the independent implementation (value not printed)
  startup   baseline-vs-overhead correlation, the phase's apps: matches the independent implementation (value not printed)
         app  control_j  freerasp_j  delta_j  delta_pct  window
  calculator      92.13      112.94    20.82      22.60 startup
  documenter      55.97       80.36    24.39      43.58 startup
       itsok      68.09       92.75    24.66      36.21 startup
 keep_recipe      77.25      102.24    25.00      32.36 startup
     linkhub      94.31      114.59    20.27      21.50 startup
    medtimer     224.60      233.87     9.27       4.13 startup
   pdfviewer      92.07      113.32    21.26      23.09 startup
poetskingdom      70.81       94.18    23.36      32.99 startup
     tipuous      69.07       88.65    19.58      28.35 startup

  window                  subset  n_apps  mean_delta_j  min_delta_j  max_delta_j  cv_joules_pct  cv_percent_pct
w

## §6.3 (RQ2): medtimer, the one freeRASP build without the INTERNET permission

medtimer's own manifest removes INTERNET (`tools:node="remove"`, upstream commit 43719af5, "Explicitly remove network
permissions"; its README describes the app as offering "full offline privacy"), and the freeRASP build kept that removal, so it is the one freeRASP build that cannot open a socket. It also
has the smallest startup overhead. Is the small overhead the missing transmission? Two checks:
(1) whether the other eight applications' overhead, which shrinks slightly as their own startup grows, already predicts
medtimer's value from its much larger control startup, without any role for the permission; and (2) the size of the gap.
One application cannot separate the two explanations; this shows whether the permission is *needed* to explain it.

In [9]:
from scipy import stats as _st
par = pd.read_csv(T.DATA / "inputs" / "variant_parity.csv")
fr = par[(par["variant"] == "freerasp_debug") & par["app"].isin(series["app"].unique())]
no_net = sorted(fr.loc[~fr["declares_internet"].astype(bool), "app"])
print("freeRASP builds without the INTERNET permission:", no_net)
st = pd.read_csv(T.TABLES / "freerasp_fixed_cost_per_app.csv").query("window == 'startup'").set_index("app")
e, m = st.drop(index="medtimer"), st.loc["medtimer"]
lr = _st.linregress(e["control_j"], e["delta_j"])
x0, n = m["control_j"], len(e)
pred = lr.intercept + lr.slope * x0
resid = e["delta_j"] - (lr.intercept + lr.slope * e["control_j"])
se = np.sqrt((resid ** 2).sum() / (n - 2)) * np.sqrt(1 + 1 / n + (x0 - e["control_j"].mean()) ** 2
                                                     / ((e["control_j"] - e["control_j"].mean()) ** 2).sum())
t = _st.t.ppf(0.975, n - 2)
out = pd.DataFrame([dict(medtimer_overhead_j=m["delta_j"], medtimer_control_j=x0,
                         eight_apps_control_j_max=e["control_j"].max(), eight_apps_overhead_j_mean=e["delta_j"].mean(),
                         gap_to_eight_mean_j=e["delta_j"].mean() - m["delta_j"],
                         eight_apps_slope_j_per_j=lr.slope, slope_p=lr.pvalue,
                         predicted_from_control_size_j=pred, prediction_lo=pred - t * se, prediction_hi=pred + t * se)])
# how much the prediction depends on any one of the eight: refit without each in turn
loo = [np.polyval(np.polyfit(e.drop(index=k)["control_j"], e.drop(index=k)["delta_j"], 1), x0) for k in e.index]
out["predicted_leave_one_app_out_min_j"], out["predicted_leave_one_app_out_max_j"] = min(loo), max(loo)
out.to_csv(T.TABLES / "freerasp_medtimer_check.csv", index=False)
T.register("tables/freerasp_medtimer_check.csv", "REPO", "6.3 RQ2 / Threats (freeRASP network)",
           "medtimer (no INTERNET): its overhead vs. what the other eight apps' trend predicts from its control size", NB)
T.record("medtimer_check", no_internet_builds=no_net, medtimer_overhead_j=m["delta_j"],
         gap_to_eight_mean_j=e["delta_j"].mean() - m["delta_j"], predicted_from_control_size_j=pred,
         prediction_interval_j=[pred - t * se, pred + t * se], prediction_leave_one_app_out_j=[min(loo), max(loo)],
         slope_p=lr.pvalue,
         note="extrapolation 2.4x beyond the eight apps' control range; one case; not separable from the permission")
print(out.round(2).T.to_string())

freeRASP builds without the INTERNET permission: ['medtimer']
                                        0
medtimer_overhead_j                  9.27
medtimer_control_j                 224.60
eight_apps_control_j_max            94.31
eight_apps_overhead_j_mean          22.42
gap_to_eight_mean_j                 13.14
eight_apps_slope_j_per_j            -0.09
slope_p                              0.13
predicted_from_control_size_j        9.27
prediction_lo                       -9.97
prediction_hi                       28.50
predicted_leave_one_app_out_min_j    5.91
predicted_leave_one_app_out_max_j   11.10


## Figure 2: freeRASP's startup overhead, the same joules at different percentages

One row per application. The dot is how much more startup energy the freeRASP build used than the unprotected control,
**in joules**; the column on the right gives the same overhead **as a percentage of that application's control**. Eight
of nine applications sit at 19.6–25.0 J while their percentages range from 21% to 44%: the same amount on top of
startups of different size, which is what a fixed cost looks like. medtimer, whose control startup is far larger, is
the exception. Only differences from the control are plotted; the control is the labelled zero line.

In [10]:
import matplotlib.transforms as mtransforms
ft = (pd.read_csv(T.TABLES / "freerasp_fixed_cost_per_app.csv").query("window == 'startup'")
      .sort_values("delta_j").reset_index(drop=True))          # largest overhead at the top
c = T.COLOURS["freeRASP"]
y = np.arange(len(ft))
fig, ax = plt.subplots(figsize=(T.FIG_SINGLE, 2.4), layout="constrained")
ax.scatter(ft["delta_j"], y, s=14, color=c, zorder=3)
T.control_line(ax, "x", outside=True, label="unprotected\ncontrol")
# the same overhead as a percentage of each application's control, as a column right of the plot
col = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
for yi, pct in zip(y, ft["delta_pct"]):
    ax.text(1.04, yi, f"+{pct:.0f}%", transform=col, ha="left", va="center", clip_on=False)
ax.annotate("% vs.\ncontrol", xy=(1.04, 1), xycoords="axes fraction", xytext=(0, 2),
            textcoords="offset points", ha="left", va="bottom", fontstyle="italic", color="0.25",
            annotation_clip=False)
ax.set_xlim(-0.08 * ft["delta_j"].max(), 1.08 * ft["delta_j"].max())
ax.set_ylim(-0.6, len(ft) - 0.4)
ax.set_yticks(y); ax.set_yticklabels(ft["app"])
ax.set_xlabel("Extra passive-phase energy vs. control (J)")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
T.register(T.save_figure(fig, "figure2_freerasp_startup_overhead", T.FIG_SINGLE), "REPO", "6.3 RQ2 (phases)",
           "freeRASP passive-phase overhead vs. control per application in joules (dots), with the same overhead as % of the control (right column)", NB)

In [11]:
# Section 5.2: one measured run, to scale, from the emitted timings (medians; the control's for the launch events).
import matplotlib.patches as mpatches
_tl_sw = pd.read_csv(T.TABLES / "startup_window_composition.csv")
_tl_ev = lambda ev: float(_tl_sw[_tl_sw["event"].str.startswith(ev) & (_tl_sw["technique"] == "Control")]["median_s"].iloc[0])
_tl_proc, _tl_disp = _tl_ev("cold launch: application process starts"), _tl_ev("cold launch: first display")
_tl_ui, _tl_warm, _tl_first = _tl_ev("Appium's UiAutomator2 server starts"), _tl_ev("Appium sends the launch intent"), _tl_ev("first scenario")
_tl_wd = pd.read_csv(T.TABLES / "workload_duration.csv").set_index("app")
_tl_work = float(_tl_wd.loc["all nine", "median_workload_s"])
_tl_lo, _tl_hi = _tl_wd.drop("all nine")["median_workload_s"].agg(["min", "max"])
_tl_lt = pd.read_csv(T.TABLES / "launch_time.csv")
_tl_launch = float(_tl_lt[_tl_lt["device"] == "all devices"]["control_mean_ms"].iloc[0]) / 1000
_tl_end = _tl_first + _tl_work
assert _tl_proc < _tl_disp < 0 < _tl_ui < _tl_warm < _tl_first < _tl_end

fig, ax = plt.subplots(figsize=(T.FIG_DOUBLE, 2.05), layout="constrained")
for _a, _b, _fc in ((_tl_proc, _tl_disp, "0.30"), (_tl_disp, 0.0, "0.82"),
                    (0.0, _tl_first, "#a6cee3"), (_tl_first, _tl_end, "#1f78b4")):
    ax.add_patch(mpatches.Rectangle((_a, 0), _b - _a, 1, facecolor=_fc, edgecolor="0.25", lw=0.6))
ax.text(_tl_first / 2, 0.5, f"Passive phase\n≈ {_tl_first:.0f} s", ha="center", va="center", fontsize=8)
ax.text((_tl_first + _tl_end) / 2, 0.5,
        f"Active phase\n≈ {_tl_work:.0f} s ({_tl_lo:.0f}–{_tl_hi:.0f} s by application)",
        ha="center", va="center", fontsize=8, color="white")
for _x, _txt, _ha in ((0.0, "Profilers start\n(t = 0)", "left"),
                      (_tl_first, "First scenario\nstarts", "left"),
                      (_tl_end, "Last scenario\nends", "right")):
    ax.plot([_x, _x], [1, 1.45], color="0.25", lw=0.6, ls=":")
    ax.text(_x + (0.8 if _ha == "left" else -0.8), 1.5, _txt, ha=_ha, va="bottom")
_tl_line = dict(arrowstyle="-", lw=0.5, color="0.25", shrinkA=0, shrinkB=0)
ax.annotate("Fresh install\nbefore every run", xy=(_tl_proc - 0.6, 0.5), xytext=(_tl_proc - 4, 0.5),
            ha="right", va="center", arrowprops=dict(arrowstyle="->", lw=0.5, color="0.25", shrinkA=0, shrinkB=0))
ax.annotate(f"Launch ≈ {_tl_launch:.1f} s\n(timed: time to first display)", xy=((_tl_proc + _tl_disp) / 2, 1),
            xytext=(_tl_proc - 4, 1.5), ha="right", va="bottom", arrowprops=_tl_line)
ax.annotate(f"Launch phase (not measured in energy):\nthe harness starts the profilers, ≈ {-_tl_disp:.0f} s", xy=(_tl_disp / 2, 0),
            xytext=(_tl_proc - 4, -0.5), ha="right", va="top", arrowprops=_tl_line)
ax.annotate(f"Appium connects (≈ {_tl_ui:.1f} s) and sends the launch intent\nagain (≈ {_tl_warm:.1f} s; five of the eight apps of the passive phase); no new process",
            xy=((_tl_ui + _tl_warm) / 2, 0), xytext=(6, -0.5), ha="left", va="top", arrowprops=_tl_line)
ax.set_xlim(-50, _tl_end + 4)
ax.set_ylim(-1.55, 2.35)
ax.set_yticks([])
ax.set_xticks([round(_tl_proc, 1), 0, 30, 60, 90, 120, round(_tl_end)])
ax.set_xticklabels([f"{round(_tl_proc, 1):g}".replace("-", "−"), "0", "30", "60", "90", "120", f"{round(_tl_end)}"])
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_bounds(_tl_proc, _tl_end)
ax.set_xlabel("Time from the first battery sample (s)")
T.register(T.save_figure(fig, "timeline_one_run", T.FIG_DOUBLE), "REPO", "5.2 Energy measurement (what the measured windows contain)",
           "one measured run to scale: fresh install, launch (timed), profiler start, passive phase, active phase; "
           "medians from startup_window_composition, workload_duration and launch_time", NB)
